# Time-varying baseline hazard (splines)

Notebook 01 used a **constant** baseline hazard $h_0$: the exponential model says
risk never changes with time-on-study. That is often too rigid. Real hazards
*move* — an infection risk that spikes early and fades, a wear-out risk that climbs
with age.

A convenient testbed is the **Weibull** baseline hazard
$$h_0(t) = \frac{k}{\lambda}\left(\frac{t}{\lambda}\right)^{k-1},$$
which rises when the shape $k>1$ and falls when $k<1$. We generate data with a
rising Weibull baseline and then fit a Cox model whose baseline hazard is a
**learnable linear spline** — flexible enough to recover the time-shape without
being told it is Weibull. The proportional-hazards structure is unchanged:
$h(t\mid x) = h_0(t)\,\exp(x\cdot\beta)$, but now $h_0(t)$ varies with $t$.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

from torchpp import IntensityBasedTPP, TokenRoles
from torchpp.utils import LinearSpline
from survival_tools import (
    make_weibull_ph, survival_batch, covariate_batch, fit_full_batch,
    model_hazard_curve, model_survival_curve, kaplan_meier,
    plot_hazard, plot_coefficients, plot_survival_km_vs_model,
)

# survival_tools selects the Agg backend on import; switch to inline afterwards
# so the figures render in the notebook.
%matplotlib inline
_ = torch.manual_seed(0)

## 1. Time-varying survival data

`make_weibull_ph` samples event times by **inverse-transform sampling**: it draws
a uniform $U$, forms the target cumulative baseline hazard
$H_0(T) = -\log U / \exp(x\cdot\beta)$, and inverts $H_0$ to get the event time
$T$. For the Weibull this inverse is closed-form. We use shape $k=2$, so the true
baseline hazard rises **linearly** in $t$. The generator hands back the true
$h_0(t)$ as a callable in `meta["h0_true"]`.

In [ ]:
BETA_TRUE = torch.tensor([0.7, -0.4])
SHAPE_K, SCALE_LAMBDA, T_OBS, N = 2.0, 2.0, 4.0, 4000

seqs, meta = make_weibull_ph(
    n=N, beta=BETA_TRUE, shape_k=SHAPE_K, scale_lambda=SCALE_LAMBDA, T_obs=T_OBS, seed=0,
)
batch = survival_batch(seqs)

censoring = 1.0 - meta["observed"].float().mean()
print(f"n = {N}   censoring rate = {censoring:.1%}")
print(f"max observed event time = {float(meta['event_time'][meta['observed']].max()):.2f}")

# the true baseline hazard rises with time (Weibull, k=2)
t = np.linspace(0.0, T_OBS, 200)
plt.figure(figsize=(6, 4))
plt.plot(t, meta["h0_true"](t).numpy(), "k")
plt.xlabel("time"); plt.ylabel("true baseline hazard h0(t)")
plt.title("Rising Weibull baseline (k=2)")
plt.show()

## 2. Cox model with a spline baseline hazard

The baseline hazard is a **linear spline** with fixed knot locations and learnable
heights. To keep the hazard positive we store raw parameters and pass them through
`softplus`. `torchpp.utils.LinearSpline` gives us both pieces we need:

- `interpolate` — the hazard value $h_0(t)$ at query times (for the intensity);
- `integrate` — the cumulative baseline hazard $H_0(t)=\int_0^t h_0$ from 0 to $t$
  (for the closed-form cumulative intensity).

**Knot placement.** The knots start at $t=0$ (the spline is undefined before its
first knot) and the last knot sits at the window end `T_obs`. Beyond the last knot
the spline extrapolates **flat** — a constant hazard — which is why the last knot
must reach the end of the data.

In [ ]:
class SplineCoxTPP(IntensityBasedTPP):
    '''Cox proportional hazards with a flexible, time-varying baseline hazard.

    Baseline hazard h0(t) is a linear spline with learnable positive heights at
    fixed knots; the covariate effect exp(x . beta) multiplies it (proportional
    hazards).  Beyond the last knot the spline extrapolates flat.
    '''

    def __init__(self, n_features, knot_locs):
        super().__init__(D=1, terminating=True, token_roles=TokenRoles(has_origin=True))
        knot_locs = torch.as_tensor(knot_locs, dtype=torch.float32)
        assert float(knot_locs[0]) == 0.0, "first knot must be at t=0"
        self.register_buffer("knot_locs", knot_locs)                 # (K,) fixed
        self.raw_h = nn.Parameter(torch.zeros(1, knot_locs.shape[0]))  # (1, K) raw heights
        self.beta = nn.Parameter(torch.zeros(n_features))            # (n_features,)

    def _heights(self):
        # Strictly-positive baseline-hazard heights at the knots.  (1, K).
        return F.softplus(self.raw_h)

    def _intensities_at_queries(self, query_times, ts):
        # h0(t) from the shared spline, times the per-subject covariate effect.  (B, Q, 1).
        B, Q = query_times.shape
        h0 = LinearSpline.interpolate(self.knot_locs, self._heights(), query_times.reshape(-1))
        h0 = h0.view(B, Q, 1)                                        # (B, Q, 1)
        mult = torch.exp(ts.covariates["x"] @ self.beta)            # (B,)
        return h0 * mult.view(B, 1, 1)

    def _closed_form_ci(self, ts, T_low, T):
        # Lambda over [T_low or 0, T] = H0(hi) * exp(x . beta), H0 the integrated spline.
        hi = self._effective_T(T, ts)                               # (B, 1)
        H = LinearSpline.integrate(self.knot_locs, self._heights(), hi)  # (B, 1)
        if T_low is not None:
            lo = self._effective_T(T_low, ts)
            H = H - LinearSpline.integrate(self.knot_locs, self._heights(), lo)
        mult = torch.exp(ts.covariates["x"] @ self.beta)            # (B,)
        return H * mult.view(-1, 1)                                 # (B, 1)

We place 8 knots evenly across the observation window.

In [ ]:
knots = torch.linspace(0.0, T_OBS, 8)
model = SplineCoxTPP(n_features=2, knot_locs=knots)
print("knots:", [round(float(k), 3) for k in knots])

## 3. Fit by maximum likelihood

In [ ]:
history = fit_full_batch(model, batch, steps=1500, lr=0.05, log_every=300)
print(f"\nfinal loss (NLL/event) = {history['losses'][-1]:.4f}")

## 4. Recover the baseline hazard and the coefficients

**A note on predictive queries.** Hazard and survival curves must be evaluated on
an *empty-history* batch — a subject carrying only its covariates, with last-event
time 0 — so any query time $t\ge 0$ is valid. `model_hazard_curve` and
`model_survival_curve` build exactly that; do not call them on the training batch,
whose subjects already have their terminal event.

To read the **baseline** hazard $h_0(t)$ we query the hazard at $x=0$, where the
covariate multiplier $\exp(x\cdot\beta)=1$, so the curve is $h_0(t)$ itself. The
spline (solid) should track the true Weibull baseline (dashed).

In [ ]:
beta_hat = model.beta.detach()
print("true beta      :", [round(float(b), 3) for b in BETA_TRUE])
print("recovered beta :", [round(float(b), 3) for b in beta_hat])

t_grid = np.linspace(0.0, T_OBS, 200)
h0_learned = model_hazard_curve(model, torch.zeros(1, 2), t_grid).numpy()[0]
h0_true = meta["h0_true"](t_grid).numpy()
mae = np.abs(h0_learned - h0_true).mean()
print(f"mean |h0_learned - h0_true| over the grid = {mae:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
plot_hazard(t_grid, h0_learned, h_true=meta["h0_true"], ax=axes[0])
axes[0].set_title("baseline hazard h0(t)")
plot_coefficients(BETA_TRUE, beta_hat, ax=axes[1])
axes[1].set_title("coefficients")
plt.tight_layout()
plt.show()

## 5. Predicted survival vs Kaplan-Meier

Averaged over the covariates of all subjects, the model's survival curve should
follow the non-parametric Kaplan-Meier estimate of the whole cohort. (The curve at
the mean covariate $x=0$ is a different quantity: survival is not linear in $x$.)

In [ ]:
obs_time = torch.minimum(meta["event_time"], torch.tensor(T_OBS)).numpy()
km_t, km_S = kaplan_meier(obs_time, meta["observed"].numpy())
S_model = model_survival_curve(model, meta["x"], t_grid).mean(dim=0).numpy()  # cohort average

fig = plot_survival_km_vs_model(km_t, km_S, t_grid, S_model, labels=["cohort"])
plt.show()

## Takeaways

- Swapping the constant baseline for a **linear-spline** baseline lets the model
  learn the *shape* of the hazard over time — here it recovers the rising Weibull
  baseline — while keeping the proportional-hazards coefficients $\beta$.
- `LinearSpline.interpolate` / `integrate` supply the two ingredients an
  `IntensityBasedTPP` needs: the intensity $\lambda(t)=h_0(t)\,e^{x\cdot\beta}$ and
  its closed-form cumulative integral $\Lambda$.
- **Caveat: flat extrapolation past the last knot.** Beyond the final knot the
  spline is constant, and the tail knots are estimated from few surviving subjects,
  so the learned $h_0(t)$ is least reliable near the end of the window. Place the
  last knot at the end of the data and read the tail with caution.